# M35 lab solution: specialist retrievers

**Brief.** Replace the single router from M30 with a supervisor over three specialist RAG agents and compare routing accuracy and cost.

This notebook builds both systems over the same three knowledge bases (`../data/kb/billing`, `technical`, `general`) and runs them on the 40 labelled tickets in `../data/tickets.jsonl`:

1. **Single router.** One structured-output call picks a knowledge base, then one call answers from the top passages of that base.
2. **Supervisor.** A LangGraph supervisor (as in notebook `04a_supervisor_pattern.ipynb`) sends the ticket to one of three specialist RAG agents, reads the answer, and either finishes or tries another specialist, with a hard cap of three turns.

Routing accuracy compares the first knowledge base each system picks with the ticket's `route` label. Cost is counted in model calls and tokens per ticket; multiply by your provider's current price to get money.

## Setup

In [ ]:
import json
import operator
import sys
import time
from pathlib import Path
from typing import Annotated, Literal

from dotenv import load_dotenv
from langchain_core.callbacks import UsageMetadataCallbackHandler
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

load_dotenv()
sys.path.insert(0, str(Path("../../../../src").resolve()))
from helpers.utils import get_experientiallabs_llm  # noqa: E402
from lab_kit import load_handbook, load_jsonl, tfidf_retriever  # noqa: E402

DATA = Path("../data")
ROUTES = ("billing", "technical", "general")
llm = get_experientiallabs_llm()
MODEL = llm.model_name
print("model:", MODEL)

## The three knowledge bases

Each specialist gets its own TF-IDF retriever over its own folder (`lab_kit.tfidf_retriever`, top 3 paragraphs). Retrieval is offline, so the cost difference between the two systems comes only from model calls.

In [ ]:
def paragraphs(folder: Path) -> list[str]:
    return [p.strip() for page in load_handbook(folder) for p in page["text"].split("\n\n") if p.strip()]


retrievers = {r: tfidf_retriever(paragraphs(DATA / "kb" / r), k=3) for r in ROUTES}
tickets = load_jsonl(DATA / "tickets.jsonl")
print({r: len(paragraphs(DATA / "kb" / r)) for r in ROUTES}, "paragraphs;", len(tickets), "tickets")

In [ ]:
ANSWER_PROMPT = (
    "You are Larkspur Cloud's {route} support agent. Answer the customer's ticket using only the passages. "
    "If the passages do not answer it, reply exactly NOT_IN_KB.\n\nPassages:\n{passages}"
)


def specialist(route: str, ticket: str, callbacks: list) -> str:
    passages = "\n---\n".join(retrievers[route](ticket)) or "(no passages)"
    msg = llm.invoke(
        [SystemMessage(ANSWER_PROMPT.format(route=route, passages=passages)), HumanMessage(ticket)],
        config={"callbacks": callbacks},
    )
    return msg.content


class Meter(UsageMetadataCallbackHandler):
    """Token usage per model (from the parent class) plus a count of model calls."""

    def __init__(self):
        super().__init__()
        self.calls = 0

    def on_llm_end(self, response, **kwargs):
        super().on_llm_end(response, **kwargs)
        self.calls += 1

    @property
    def tokens(self) -> int:
        return sum(u.get("total_tokens", 0) for u in self.usage_metadata.values())

## System 1: the single router

In [ ]:
class Route(BaseModel):
    route: Literal["billing", "technical", "general"] = Field(
        description="billing: invoices, payments, refunds, service credits. technical: servers, outages, backups, data export, "
        "security, API limits, regions. general: plans, sizes, support tiers, onboarding, account deletion."
    )


router_llm = llm.with_structured_output(Route)


def single_router(ticket: str) -> dict:
    cb = Meter()
    route = router_llm.invoke(
        [SystemMessage("Pick the knowledge base that answers this support ticket."), HumanMessage(ticket)],
        config={"callbacks": [cb]},
    ).route
    answer = specialist(route, ticket, [cb])
    return {"first_route": route, "routes": [route], "answer": answer, "calls": cb.calls, "tokens": cb.tokens}


print(single_router(tickets[0]["text"]))

## System 2: the supervisor over three specialist agents

The supervisor sees the ticket and every specialist answer so far, and returns either a specialist name or `FINISH`. A specialist that answers `NOT_IN_KB` sends control back, so the supervisor can try another base. `MAX_TURNS = 3` is enforced in the router function, not in the prompt.

In [ ]:
MAX_TURNS = 3


class Decision(BaseModel):
    next: Literal["billing", "technical", "general", "FINISH"] = Field(
        description="The specialist to ask next, or FINISH when an answer already resolves the ticket."
    )


SUPERVISOR_PROMPT = (
    "You supervise three Larkspur Cloud support specialists: billing (invoices, payments, refunds, service credits), "
    "technical (servers, outages, backups, data export, security, API limits, regions) and general (plans, sizes, support tiers, "
    "onboarding, account deletion). Pick the specialist for the ticket. When a specialist answer resolves "
    "it, reply FINISH. Never ask the same specialist twice."
)


class SupState(TypedDict):
    ticket: str
    routes: Annotated[list[str], operator.add]
    answers: Annotated[list[str], operator.add]
    next: str


def make_supervisor(cb: Meter):
    decide = llm.with_structured_output(Decision)

    def supervisor(state: SupState) -> dict:
        history = "\n".join(f"{r}: {a}" for r, a in zip(state["routes"], state["answers"])) or "(none yet)"
        d = decide.invoke(
            [SystemMessage(SUPERVISOR_PROMPT), HumanMessage(f"Ticket: {state['ticket']}\n\nAnswers so far:\n{history}")],
            config={"callbacks": [cb]},
        )
        return {"next": d.next}

    def node(route: str):
        def run(state: SupState) -> dict:
            return {"routes": [route], "answers": [specialist(route, state["ticket"], [cb])]}
        return run

    def after_supervisor(state: SupState) -> str:
        if state["next"] == "FINISH" or len(state["routes"]) >= MAX_TURNS or state["next"] in state["routes"]:
            return END
        return state["next"]

    g = StateGraph(SupState)
    g.add_node("supervisor", supervisor)
    for r in ROUTES:
        g.add_node(r, node(r))
        g.add_edge(r, "supervisor")
    g.add_edge(START, "supervisor")
    g.add_conditional_edges("supervisor", after_supervisor, [*ROUTES, END])
    return g.compile()


def supervised(ticket: str) -> dict:
    cb = Meter()
    out = make_supervisor(cb).invoke(
        {"ticket": ticket, "routes": [], "answers": [], "next": ""},
        config={"recursion_limit": 2 * MAX_TURNS + 3},
    )
    answer = next((a for a in reversed(out["answers"]) if "NOT_IN_KB" not in a), out["answers"][-1] if out["answers"] else "")
    first = out["routes"][0] if out["routes"] else "none"
    return {"first_route": first, "routes": out["routes"], "answer": answer, "calls": cb.calls, "tokens": cb.tokens}


print(supervised(tickets[0]["text"]))

## Run both systems on the 40 tickets

In [ ]:
rows = []
for t in tickets:
    for name, system in (("single router", single_router), ("supervisor", supervised)):
        start = time.perf_counter()
        r = system(t["text"])
        rows.append({"id": t["id"], "system": name, "label": t["route"], "seconds": time.perf_counter() - start, **r})
    print(t["id"], t["route"], "|", rows[-2]["first_route"], "|", rows[-1]["routes"])

## Results

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
df["correct"] = df["first_route"] == df["label"]
summary = df.groupby("system").agg(
    routing_accuracy=("correct", "mean"),
    calls_per_ticket=("calls", "mean"),
    tokens_per_ticket=("tokens", "mean"),
    seconds_per_ticket=("seconds", "mean"),
    specialists_per_ticket=("routes", lambda s: s.map(len).mean()),
)
# Fill in your provider's current blended price per million tokens to turn tokens into money.
PRICE_PER_MILLION_TOKENS_USD = None
if PRICE_PER_MILLION_TOKENS_USD:
    summary["usd_per_ticket"] = summary["tokens_per_ticket"] * PRICE_PER_MILLION_TOKENS_USD / 1e6
print("model:", MODEL, "| run:", time.strftime("%Y-%m-%d"))
print(summary.round(3).to_markdown())

In [ ]:
# Routing accuracy per label, and the tickets each system sent to the wrong base first.
print(df.pivot_table(index="label", columns="system", values="correct", aggfunc="mean").round(2).to_markdown())
print(df[~df["correct"]][["id", "system", "label", "first_route", "routes"]].to_markdown(index=False))

## What this shows

- Both systems make one routing decision before any retrieval, so their first-choice accuracy measures the same thing: whether a model can map a ticket to a knowledge base from its wording.
- The supervisor costs at least one extra call per ticket (the closing `FINISH` decision) and more when a specialist answers `NOT_IN_KB` and it tries another base. That retry is what it buys: a ticket the router misroutes can still end with an answer.
- Compare `specialists_per_ticket` with the misrouted tickets above: if the supervisor rarely calls a second specialist, the extra cost buys little on this ticket set, and the single router is the better default.